<a href="https://colab.research.google.com/github/edwardoughton/spatial_computing/blob/main/7_01_Scripting_Spatial_Processing_via_GeoPandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7: Spatial processing with GeoPandas — Part 1

Last week we introduced Pandas and GeoPandas. Today we will download spatial data from a server and build a processing workflow that produces findings about Virginia's administrative areas.

We will cover reading and checking data, projecting it, calculating area, filtering rows, mapping results, exporting layers, creating boundaries and centroids, and converting between GeoDataFrames and Python records.

**Question:** How do the areas of Virginia counties and independent cities compare?

By the end, submit a map, a summary CSV, a spatial output, and three sentences describing a finding and a limitation. These are administrative polygons, not measurements of population or service access.

Run examples in order. Task cells are for your own code. If GeoPandas is unavailable, run `%pip install geopandas matplotlib` in a separate cell and retry.


## Recap on `GeoPandas`

`GeoPandas` is a very popular spatial data processing package based on the popular `Pandas` library.

Essentially, `GeoPandas` provides us with a range of useful spatial functions enabling us to swiftly analyze and manipulate spatial data via a dataframe structure.

* The key data structure within `GeoPandas` is a `GeoDataFrame` which essentially stores our spatial data in a `Pandas` dataframe, only with an additional geometry column containing a `Shapely` object.
* A range of spatial operations are thus available to us for use via our `GeoDataFrame` structure, including overlaying geometries, spatial joins, and geometry buffering. Via these tools we can quantify relationships between different geospatial entities, using these results for a wide range of applications (e.g., writing scientific papers, decision-support modeling etc.).
* We can easily handle importing and exporting spatial data thanks to the range of functionalities `GeoPandas` provides. This includes importing/exporting geospatial data via Shapefile, GeoJSON, GeoPackage etc. This reduces the number of lines of additional code we need to allocate to these more basic/vanilla tasks.
* Other libraries may have useful functions we want to use, and `GeoPandas` provides us with the ability to seamlessly integrate with other popular geospatial libraries. For example, we can utilize geometric operations available within Shapely and readily apply them to the geometries within our dataframe.
* Finally, `GeoPandas` does provide different plotting capabilities for visualizing geospatial data directly from a `GeoDataFrame`. Just be aware this will require Matplotlib for more basic plotting functions, or utilizing more niche libraries for more advanced mapping.

Let us get into exploring more of these capabilities using actual coding examples.



## `GeoPandas` types and subclasses

`GeoPandas` implements `GeoSeries` and `GeoDataFrame` structures, which are both subclasses of `pandas.Series` and `pandas.DataFrame`. Geometries are handled in the `geopandas.GeoSeries`, via shapely.

Therefore, your `GeoDataFrame` is a combination of `pandas.Series`, with traditional data (numerical, boolean, text etc.), and `geopandas.GeoSeries`, with geometries (points, polygons etc.).


## Download spatial data from a server

Use the same Virginia dataset as Week 6. The HTTPS URL points to a fixed course-data version. A raw-data URL returns a file rather than a GitHub webpage.

Here we explicitly download and save the ZIP before reading it. This gives us a reusable local copy and separates acquisition from processing. The archive contains the shapefile and its companion `.dbf`, `.shx`, `.prj`, and `.cpg` files.

An internet connection is required for this cell. Keep the source URL with your results. See [GeoPandas reading and writing files](https://geopandas.org/en/stable/docs/user_guide/io.html).


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

data_url = 'https://raw.githubusercontent.com/edwardoughton/spatial_computing/510dcb3b7d6954999434df162b077dabe2a98891/data/virginia.zip'
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
data_path = data_dir / 'virginia.zip'
urlretrieve(data_url, data_path)
print(f'Downloaded {data_path.stat().st_size:,} bytes from {data_url}')


Read the saved ZIP with `gpd.read_file()`. GeoPandas reads the source CRS from the dataset; do not override it on import. Inspect the returned records before doing calculations.


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

source_gdf = gpd.read_file(data_path)
if source_gdf.crs is None:
    raise ValueError('The source CRS is missing; consult the data metadata before proceeding.')
print('Features:', len(source_gdf), '| CRS:', source_gdf.crs)
print(source_gdf.geom_type.value_counts())
print('Missing geometries:', source_gdf.geometry.isna().sum())
print('Empty geometries:', source_gdf.geometry.is_empty.sum())
print('Invalid geometries:', (~source_gdf.geometry.is_valid).sum())
source_gdf.head(3)


You might find it easier to subset your `GeoDataframe`, which you can easily do the same way that we utilized in `Pandas`.


In [ ]:
gdf = source_gdf[['GID_2', 'geometry', 'NAME_1', 'NAME_2', 'TYPE_2']].copy()
gdf.head(3)


## Task: inspect before processing

Print the column names, administrative types, and number of unique `GID_2` values. Does each row have a unique identifier? Compare the row count with Week 6. What can you learn from the geometry checks, and what do they leave unknown?


## Managing coordinate reference systems

The `.crs` property reports the current CRS. Longitude and latitude are measured in degrees. Use a suitable projected CRS before calculating area or centroids.


In [ ]:
# Example: Checking the crs of our dataframe
gdf.crs

In [ ]:
# Example: Plotting the data to view crs units via `.plot()`
gdf.plot()

Use `.to_crs()` to transform coordinates. Here we use EPSG:3968 (NAD83 / Virginia Lambert), a projected CRS designed for Virginia with meter units. Web Mercator (EPSG:3857) distorts area and is unsuitable for these comparisons.

`.set_crs()` assigns a known CRS label; it does not transform coordinates. Never guess the source CRS.


In [ ]:
gdf = gdf.to_crs('EPSG:3968')
gdf.head(3)


In [ ]:
# Example: Check changed crs
gdf.crs

In [ ]:
# Example: Plotting the data to view crs units via `.plot()`
gdf.plot()

## Creating a new column

Next, we can consider how we would create a new column in a `GeoPandas` `GeoDataFrame`.

For example, we could just create a single column with a pre-defined value for all columns (the most straight-forward example):

In [ ]:
# Example: Adding a new column
gdf['new_column'] = 'I_am_a_string'
gdf['new_column2'] = 53.5
gdf['new_column3'] = 0
gdf['new_column4'] = "(2.334, 4.653)"
gdf.head(3)

Calculate each polygon's area with `.geometry.area`. The projected coordinates have meter units, so the resulting area is in square meters. Reprojection changes geometry but does not recalculate existing numeric area columns.


In [ ]:
# Example: Finding the area of our polygons in square meters
gdf['area_m2'] = gdf['geometry'].area
gdf.head(3)

If we want to then create a new column with the area in square kilometers, we would divide by 1x10^6.

In [ ]:
# Example: Finding the area of our polygons in square kilometers
gdf['area_km2'] = gdf['area_m2'] / 1e6
gdf.head(3)

## Plotting

`GeoPandas` provides an easy plotting interface via the `GeoDataFrame.plot()` function.

To colorcode the map, we can pass the column of interest, like so:

In [ ]:
ax = gdf.plot(column='area_km2', legend=True, edgecolor='gray', linewidth=0.3, figsize=(10, 6))
ax.set_title('Virginia administrative polygon area (km²)')
ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
# Optional interactive map: first run %pip install folium mapclassify
# Then uncomment the line below. The static map above needs no extra packages.
# gdf.to_crs('EPSG:4326').explore('area_km2', legend=True)


## Manipulating column data




Use an actual missing numeric value (`float('nan')`), rather than the text `'NaN'`. Work on a copy to preserve our calculated results.


In [ ]:
example_gdf = gdf.copy()
example_gdf['area_km2'] = float('nan')
example_gdf.head(3)


Fill the example column by calculating area again. The main `gdf` remains available for filtering and analysis.


In [ ]:
example_gdf['area_km2'] = example_gdf.geometry.area / 1e6
example_gdf.head(3)


# Subsetting rows


We can subset the rows of the dataframe in a variety of ways. You need to familiarize yourself with how to subset based on the different options available.

For example, what about if we wanted to subset those with an area below the mean?



In [ ]:
# Example: Subsetting rows below the mean
mean_area_km2 = gdf['area_km2'].mean()
print("The mean area of Virginia GID_2 regions is {} km2".format(mean_area_km2))

print("Currently we have {} areas (e.g., {} rows)".format(len(gdf), len(gdf)))
subset = gdf[gdf['area_km2'] < mean_area_km2]
print("Now we have {} areas (e.g., {} rows)".format(len(subset), len(subset)))
subset.head(3)

The way to understand this code:

`subset = gdf[gdf['area_km2'] < mean_area_km2]`

Is by recognizing:

* The first part to the left of the equals allocates the results to a new df called `subset`.
* The inner layer of the square parentheses evaluates whether the `area_km2` value is below the `mean_area_km2` (leading to a value for each row of either `True` or `False`).
* The outer layer subsets all rows which evaluate to `True`.

Thus, our final dataframe is the subset of the rows with an area below the mean value.

Here is an example of the result of the inner logic utilized, which evaluates to either True or False.

In [ ]:
# Example: Evaluating a piece of logic to either True or False.
gdf['area_km2'] < mean_area_km2

Now when you put this code together, you can change this subset logic to whatever you want, for example, when we have a string:

In [ ]:
# Example: Subsetting rows equalling a string
print("Currently we have {} area(s) (e.g., {} rows)".format(len(gdf), len(gdf)))
subset = gdf[gdf['NAME_2'] == "Alexandria"]
print("Now we have {} area(s) (e.g., {} rows)".format(len(subset), len(subset)))
subset.head(3)

In [ ]:
# Example: Subsetting rows equaling a string
print("Currently we have {} areas (e.g., {} rows)".format(len(gdf), len(gdf)))
subset = gdf[gdf['TYPE_2'] == "Independent City"]
print("Now we have {} areas (e.g., {} rows)".format(len(subset), len(subset)))
subset.head(3)

We can also subset rows based on a column, when they might have different values.

There are a lot of different ways to subset rows, so this is just one possible way using dataframes, via the `.isin()` function.

In [ ]:
# Example: Subsetting rows using `.isin()`
print("Currently we have {} area(s) (e.g., {} rows)".format(len(gdf), len(gdf)))
subset = gdf[gdf['NAME_2'].isin(['Bath', 'Bedford', 'Bristol', 'Richmond'])]
print("Now we have {} area(s) (e.g., {} rows)".format(len(subset), len(subset)))
subset.head(4)

## Exporting results

Export a separate selection and preserve the main GeoDataFrame. GeoPackage stores geometry, attributes, and CRS in one file. For a CSV, select the non-geometry columns. GeoJSON should be exported in EPSG:4326.


In [ ]:
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)
export_gdf = gdf[['GID_2', 'NAME_2', 'TYPE_2', 'area_km2', 'geometry']].copy()
export_gdf.to_file(output_dir / 'virginia_processed.gpkg', layer='administrative_areas', driver='GPKG')
export_gdf.drop(columns='geometry').to_csv(output_dir / 'virginia_areas.csv', index=False)


## Obtaining polygon boundaries and centroids

Create separate GeoDataFrames with each derived geometry as the active geometry column. Calculate centroids in our projected CRS. A centroid can fall outside a concave polygon; use `.representative_point()` if you need an interior point.


In [ ]:
boundaries = gdf[['GID_2', 'NAME_2', 'geometry']].copy()
boundaries['geometry'] = gdf.geometry.boundary
boundaries.head(3)


In [ ]:
centroids = gdf[['GID_2', 'NAME_2', 'geometry']].copy()
centroids['geometry'] = gdf.geometry.centroid
centroids.head(3)


## Task: derived geometries

Export boundaries and centroids into separate GeoPackage files. Plot them over the original polygons using a shared Matplotlib axis. Explain what a centroid represents and why it is not necessarily a town center.


## Converting data to and from a `GeoPandas` dataframe

Importantly, we sometimes want to convert from a dataframe to a list of dictionaries, or vice versa.

In the example below, we will import our data, convert to a list of dictionaries, and then print the contents.

The key line uses the function `.to_dict("records")` which converts the `GeoDataFrame` into a list of dicts.

In [ ]:
records_gdf = gdf[['GID_2', 'geometry']].copy()
list_of_dicts = records_gdf.to_dict('records')
for my_row in list_of_dicts[:3]:
    print(my_row)

restored_gdf = gpd.GeoDataFrame(list_of_dicts, geometry='geometry', crs=records_gdf.crs)
print('Restored records:', len(restored_gdf), '| CRS:', restored_gdf.crs)


## Task: Northern Virginia

For this task define your study area as Fairfax, Arlington, Alexandria, Loudoun, and Prince William. Inspect exact `NAME_2` labels and `TYPE_2` values first; Fairfax can refer to a county and an independent city. State which records you select.

* Filter the shapes using `.isin()` and print names, types, and row count.
* Find total, mean, median, minimum, and maximum polygon area in km².
* Map and export your selection to GeoPackage.
* Describe a finding, making clear that your total refers to the selected administrative polygons.


In [ ]:
# Enter your attempt here


## Task: Shenandoah study area

Define a county-based study area using Augusta, Rockingham, Page, Madison, Greene, Rappahannock, and Warren. Check the exact labels in this dataset, then filter and summarize their areas as above.

This measures entire administrative polygons, **not the area of Shenandoah National Park**. Explain what additional boundary dataset and spatial operation you would need to measure the park's area within each county. We will introduce intersection in Week 8.


In [ ]:
# Enter your attempt here


## Task: answer our question

Compare records categorized as `County` and `Independent City` in `TYPE_2`.

* Summarize count, sum, mean, median, minimum, and maximum `area_km2` by type.
* Export the summary to CSV and map the two types.
* Write three sentences: a numerical finding, whether mean and median tell the same story, and a limitation of using administrative area alone.

Keep the source URL and CRS with your submission. The dataset is a fixed course snapshot, so describe results as properties of these polygons rather than assuming they represent current boundaries.


In [ ]:
# Enter your attempt here
